# CHEM 269 Final Project
## 3D Conformational Descriptors and Dual-Dielectric Solvent Modeling to Decode Cyclic Peptide Membrane Permeation
**Jorge Carmona | March 2026**

---

### Running on Google Colab
Upload the full project folder to Drive at `MyDrive/chem269_final/` then run Cell 1.
The folder should contain:
```
MyDrive/chem269_final/
├── CycPeptMPDB_Peptide_All (2).csv
├── scripts/
├── data/
└── results/
    ├── conformer_descriptors_raw.csv   ← from Tier-1 Colab run
    └── tier2_reference_results.csv     ← from Tier-2 Colab run
```

### Running locally
Run from the `notebooks/` directory. ROOT is auto-detected as the project root.

In [ ]:

# ── CELL 1: Environment setup ─────────────────────────────────────────────────
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Installing packages...')
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'rdkit', 'umap-learn', 'scikit-learn-extra', 'hdbscan', 'tqdm'], check=True)
    # ── Set ROOT to your Drive folder ────────────────────────────────────────
    ROOT = Path('/content/drive/MyDrive/chem269_final')
    if not ROOT.exists():
        raise FileNotFoundError(
            f'Project folder not found at {ROOT}\n'
            f'Upload the project to MyDrive/chem269_final/ and re-run.'
        )
else:
    ROOT = Path('..').resolve()   # local: notebooks/ -> project root

DATA_CSV = ROOT / 'CycPeptMPDB_Peptide_All (2).csv'
DATA_DIR = ROOT / 'data'
RESULTS  = ROOT / 'results'
FIGURES  = RESULTS / 'figures'
SCRIPTS  = ROOT / 'scripts'

for d in [DATA_DIR, RESULTS, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SCRIPTS))

print(f'ROOT    : {ROOT}')
print(f'Colab   : {IN_COLAB}')
print(f'Scripts : {SCRIPTS.exists()}')
print(f'Data CSV: {DATA_CSV.exists()}')


In [ ]:
# ── CELL 2: Imports ───────────────────────────────────────────────────────────
import warnings, subprocess
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from IPython.display import Image, display

from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

PAMPA_THRESHOLD = -6.0
CYCLOA_IDS      = {1, 22, 932, 981, 1822, 1862, 2356, 7188, 7353}
RANDOM_STATE    = 42

def require_file(path, hint=''):
    """Raise a clear error if a required file is missing."""
    if not Path(path).exists():
        msg = f'Required file not found: {path}'
        if hint:
            msg += f'\nHint: {hint}'
        raise FileNotFoundError(msg)
    return Path(path)

def show_fig(path, fallback_msg='Figure not generated yet.'):
    """Display a figure if it exists, otherwise print a message."""
    if Path(path).exists():
        display(Image(str(path)))
    else:
        print(f'[Missing] {fallback_msg} ({path})')

def run_script(cmd, label):
    """Run a pipeline script, print stdout, raise on error."""
    print(f'Running {label}...')
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ROOT))
    if r.stdout:
        print(r.stdout[-3000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError(f'{label} failed (returncode={r.returncode})')
    print(f'{label} complete.')

print('Imports OK')

In [ ]:
# ── CELL 3: Prerequisites check ───────────────────────────────────────────────
# Run this before proceeding — it tells you exactly what is and isn't ready.

checks = [
    (DATA_CSV,                              'Raw database CSV'),
    (DATA_DIR / 'pampa_curated.csv',        'Curated PAMPA data  (run curate_data.py or Section 1)'),
    (RESULTS  / 'conformer_descriptors_raw.csv', 'Tier-1 conformer results  (download from Tier-1 Colab)'),
    (RESULTS  / 'tier2_reference_results.csv',   'Tier-2 CREST results  (download from Tier-2 Colab)'),
]

all_ok = True
for path, label in checks:
    exists = Path(path).exists()
    icon   = '✓' if exists else '✗ MISSING'
    if not exists:
        all_ok = False
    print(f'  {icon:<12}  {label}')

print()
if all_ok:
    print('All prerequisites present — ready to run.')
else:
    print('Some files are missing. The notebook will skip those sections.')
    print('Sections 1 and 2 run regardless; others require the files above.')

---
## Section 1 — Data Loading and Curation

In [ ]:
# ── CELL 4: Curate data (skips if already done) ───────────────────────────────
curated_path = DATA_DIR / 'pampa_curated.csv'

if curated_path.exists():
    print(f'Curated data already exists — skipping curation.')
else:
    require_file(DATA_CSV, 'Upload CycPeptMPDB_Peptide_All (2).csv to the project root.')
    run_script(
        [sys.executable, str(SCRIPTS / 'curate_data.py'),
         '--input', str(DATA_CSV), '--outdir', str(DATA_DIR)],
        'curate_data.py'
    )

In [ ]:
# ── CELL 5: Load and summarize PAMPA dataset ──────────────────────────────────
require_file(curated_path)
pampa = pd.read_csv(curated_path, low_memory=False)
pampa['permeable'] = (pampa['PAMPA'] >= PAMPA_THRESHOLD).astype(int)

n_total   = len(pampa)
n_perm    = pampa['permeable'].sum()
n_3dpsa   = pampa['CHCl3_3DPSA'].notna().sum()

print(f'PAMPA compounds   : {n_total:,}')
print(f'Permeable (>={PAMPA_THRESHOLD}) : {n_perm:,} ({100*n_perm/n_total:.1f}%)')
print(f'Impermeable       : {n_total-n_perm:,} ({100*(n_total-n_perm)/n_total:.1f}%)')
print(f'DB 3DPSA coverage : {n_3dpsa:,} ({100*n_3dpsa/n_total:.1f}%)')
print(f'PAMPA range       : {pampa["PAMPA"].min():.2f} to {pampa["PAMPA"].max():.2f}')

In [ ]:
# ── CELL 6: Figure 1 — PAMPA distribution + DB ΔPSA scatter ──────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Panel A: PAMPA histogram
ax1.hist(pampa['PAMPA'].dropna(), bins=60, color='steelblue', edgecolor='white', lw=0.3)
ax1.axvline(PAMPA_THRESHOLD, color='red', ls='--', lw=1.5, label=f'Threshold = {PAMPA_THRESHOLD}')
ax1.set_xlabel('PAMPA LogPexp (log cm/s)')
ax1.set_ylabel('Count')
ax1.set_title(f'PAMPA Distribution  (n={n_total:,})')
ax1.legend()

# Panel B: DB ΔPSA vs PAMPA
sub3d = pampa[pampa['CHCl3_3DPSA'].notna() & pampa['H2O_3DPSA'].notna()].copy()
sub3d['delta_3DPSA_db'] = sub3d['H2O_3DPSA'] - sub3d['CHCl3_3DPSA']

ax2.scatter(sub3d['delta_3DPSA_db'], sub3d['PAMPA'],
            s=4, alpha=0.15, c='steelblue', rasterized=True)
cycloA = sub3d[sub3d['ID'].isin(CYCLOA_IDS)]
if len(cycloA):
    ax2.scatter(cycloA['delta_3DPSA_db'], cycloA['PAMPA'],
                s=120, c='crimson', marker='*', zorder=10, label='CycloA')
ax2.axhline(PAMPA_THRESHOLD, color='red', ls='--', lw=1)
ax2.set_xlabel('DB ΔPSA = H₂O_3DPSA − CHCl₃_3DPSA  (Å²)')
ax2.set_ylabel('PAMPA LogPexp (log cm/s)')
ax2.set_title(f'DB 3D ΔPSA vs Permeability  (n={len(sub3d):,})')
ax2.legend(fontsize=9)

r, p = stats.spearmanr(sub3d['delta_3DPSA_db'], sub3d['PAMPA'])
ax2.text(0.04, 0.97, f'Spearman ρ = {r:.3f}\np = {p:.2e}',
         transform=ax2.transAxes, fontsize=9, va='top',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig(FIGURES / 'fig1_data_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2 — Tier-1 Conformer Results

Generated by `scripts/conformer_engine.py` (ETKDGv3 + MMFF94s, 20 conformers/molecule).  
**Do not re-run here** — load the CSV from the Tier-1 Colab run.

In [ ]:
# ── CELL 7: Load Tier-1 conformer descriptors ─────────────────────────────────
conf_csv = RESULTS / 'conformer_descriptors_raw.csv'

if not conf_csv.exists():
    print('Tier-1 results not found — skipping Section 2.')
    print('Download conformer_descriptors_raw.csv from the Tier-1 Colab and place in results/')
    conf_df = None
else:
    conf_df = pd.read_csv(conf_csv)
    ok = conf_df[conf_df['error'].isna()]

    print(f'Total molecules : {len(conf_df):,}')
    print(f'Successful      : {len(ok):,}  ({100*len(ok)/len(conf_df):.1f}%)')
    print(f'Failed          : {conf_df["error"].notna().sum():,}')
    if conf_df['error'].notna().any():
        print('Failure breakdown:')
        print(conf_df['error'].value_counts().to_string())
    print()

    delta_cols = [c for c in ok.columns if c.startswith('delta_') or c in ['psa3d_spread','psa3d_std','hb_spread']]
    print('Δ descriptor statistics (successful molecules):')
    print(ok[delta_cols].describe().round(2).to_string())

In [ ]:
# ── CELL 8: Tier-1 Δ descriptor distributions ─────────────────────────────────
if conf_df is None:
    print('Skipped — no Tier-1 results.')
else:
    ok = conf_df[conf_df['error'].isna()]
    plot_cols = [
        ('delta_psa3d',   'ΔPSA (Å²)',       'Max-PSA − Min-PSA conformer'),
        ('delta_hb',      'ΔHB (count)',      'Intramolecular H-bonds: mem − aq'),
        ('psa3d_spread',  'PSA spread (Å²)',  'Max − Min PSA across all conformers'),
        ('delta_Rg',      'ΔRg (Å)',          'Radius of gyration: aq − mem'),
    ]
    plot_cols = [(c, xl, t) for c, xl, t in plot_cols if c in ok.columns]

    n_plots = len(plot_cols)
    fig, axes = plt.subplots(1, n_plots, figsize=(4*n_plots, 4))
    if n_plots == 1:
        axes = [axes]

    for ax, (col, xlabel, title) in zip(axes, plot_cols):
        vals = ok[col].dropna()
        ax.hist(vals, bins=50, color='steelblue', edgecolor='white', lw=0.2)
        ax.axvline(vals.median(), color='red', ls='--', lw=1.2, label=f'Median={vals.median():.1f}')
        ax.set_xlabel(xlabel)
        ax.set_ylabel('Count')
        ax.set_title(f'{title}\n(n={len(vals):,})')
        ax.legend(fontsize=8)

    plt.suptitle('Tier-1 ETKDGv3 Δ Descriptor Distributions', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES / 'fig2_tier1_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Section 3 — Feature Matrix

In [ ]:
# ── CELL 9: Build feature matrix ──────────────────────────────────────────────
fm_path = RESULTS / 'feature_matrix.csv'

conf_arg = str(conf_csv) if (conf_csv.exists()) else ''

cmd = [
    sys.executable, str(SCRIPTS / 'build_feature_matrix.py'),
    '--pampa',    str(DATA_DIR / 'pampa_curated.csv'),
    '--outdir',   str(RESULTS),
]
if conf_arg:
    cmd += ['--conformers', conf_arg]

run_script(cmd, 'build_feature_matrix.py')

In [ ]:
# ── CELL 10: Verify feature matrix ───────────────────────────────────────────
require_file(fm_path)
fm = pd.read_csv(fm_path, low_memory=False)
print(f'Feature matrix shape : {fm.shape}')
print(f'PAMPA values         : {fm["PAMPA"].notna().sum():,}')
print()

check_cols = [
    ('MolLogP',          '2D baseline'),
    ('TPSA',             '2D baseline'),
    ('delta_3DPSA_db',   'DB Δ PSA'),
    ('H2O_3DPSA',        'DB 3DPSA aqueous'),
    ('CHCl3_3DPSA',      'DB 3DPSA membrane'),
    ('delta_psa3d',      'Tier-1 ΔPSA  ← needs Colab results'),
    ('delta_hb',         'Tier-1 ΔHB   ← needs Colab results'),
    ('psa3d_spread',     'Tier-1 PSA spread ← needs Colab results'),
]
print(f'  {"Feature":<25} {"Group":<35} {"N values":>10}')
print('  ' + '-'*72)
for col, group in check_cols:
    n = fm[col].notna().sum() if col in fm.columns else 0
    icon = '✓' if n > 100 else ('⚠' if n > 0 else '✗')
    print(f'  {icon} {col:<25} {group:<35} {n:>10,}')

---
## Section 4 — Correlation Analysis

In [ ]:
# ── CELL 11: Run correlation analysis ────────────────────────────────────────
run_script(
    [sys.executable, str(SCRIPTS / 'correlation_analysis.py'),
     '--matrix', str(fm_path), '--outdir', str(RESULTS)],
    'correlation_analysis.py'
)

In [ ]:
# ── CELL 12: Correlation and AUC results ─────────────────────────────────────
corr_df = pd.read_csv(RESULTS / 'correlation_table.csv')
auc_df  = pd.read_csv(RESULTS / 'auc_roc_table.csv')

print('=== Features ranked by Spearman ρ ===')
cols = ['Feature', 'Group', 'N', 'Pearson_r', 'Spearman_rho', 'Spearman_p']
cols = [c for c in cols if c in corr_df.columns]
print(corr_df[cols].to_string(index=False))

print()
print('=== Features ranked by AUC-ROC ===')
cols2 = ['Feature', 'Group', 'N', 'AUC_ROC']
cols2 = [c for c in cols2 if c in auc_df.columns]
print(auc_df[cols2].to_string(index=False))

In [ ]:
# ── CELL 13: Correlation figures ──────────────────────────────────────────────
for fname, title in [
    ('correlation_heatmap.png', 'Spearman ρ heatmap'),
    ('auc_roc_bar.png',         'AUC-ROC by feature'),
    ('scatter_top_features.png','Top feature scatter plots'),
]:
    print(f'--- {title} ---')
    show_fig(FIGURES / fname, f'{fname} not yet generated')

---
## Section 5 — UMAP Visualization

**Pipeline:** RobustScaler → PCA → UMAP (cosine) → Leiden clustering

- **RobustScaler**: centers to median, scales by IQR — robust to the heavy-tailed distributions of cyclic peptide descriptors
- **Silhouette scores** are computed on PCA coordinates, not 2D UMAP (UMAP distorts inter-cluster distances)

In [ ]:

# ── CELL 14: PCA elbow plots — validate dimensionality before UMAP ────────────
# RobustScaler is applied first (same as umap_visualization.py).
# PCA is run on the scaled features; elbow shows how many components
# are needed before passing to K-Medoids and UMAP.
import json

fg_path = RESULTS / 'feature_groups.json'
if fg_path.exists():
    with open(fg_path) as f:
        feature_groups = json.load(f)
else:
    feature_groups = {
        '2D_baseline':  ['MolWt','MolLogP','TPSA','NumHAcceptors',
                         'NumHDonors','NumRotatableBonds','RingCount'],
        'DB_delta':     ['delta_3DPSA_db','H2O_3DPSA','CHCl3_3DPSA'],
        'Tier1_delta':  ['delta_psa3d','delta_hb','delta_Rg','delta_NPR1',
                         'delta_NPR2','psa3d_spread','psa3d_std','hb_spread'],
    }

panels = {
    'Panel A — 2D':      feature_groups.get('2D_baseline', []),
    'Panel B — 3D Δ':    feature_groups.get('DB_delta', []) + feature_groups.get('Tier1_delta', []),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (panel_name, feats) in zip(axes, panels.items()):
    avail = [f for f in feats if f in fm.columns and fm[f].notna().sum() > 100]
    if len(avail) < 2:
        ax.text(0.5, 0.5, f'Insufficient data\n({len(avail)} features available)',
                transform=ax.transAxes, ha='center', va='center')
        ax.set_title(panel_name)
        continue

    sub = fm[avail].dropna()
    X   = RobustScaler().fit_transform(sub.values)
    n_c = min(len(avail), len(sub) - 1, 20)
    pca = PCA(n_components=n_c, random_state=RANDOM_STATE).fit(X)

    cum = np.cumsum(pca.explained_variance_ratio_) * 100
    ax.bar(range(1, len(cum)+1), pca.explained_variance_ratio_*100,
           color='steelblue', alpha=0.7)
    ax.plot(range(1, len(cum)+1), cum, 'r-o', ms=4, label='Cumulative')
    ax.axhline(90, color='grey', ls='--', lw=0.8, label='90% threshold')
    ax.set_xlabel('PCA Component')
    ax.set_ylabel('Variance Explained (%)')
    n_90 = int(np.searchsorted(cum, 90)) + 1
    ax.set_title(f'{panel_name}\nn={len(sub):,} molecules, {len(avail)} features\n'
                 f'{n_90} components → 90% variance')
    ax.legend(fontsize=8)
    print(f'{panel_name}: {len(avail)} features, {n_90} PCA components for 90%, n={len(sub):,}')

plt.suptitle('PCA Elbow — RobustScaler → PCA (input to K-Medoids + UMAP)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'pca_elbow_plots.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── CELL 15: Run UMAP visualization ──────────────────────────────────────────
run_script(
    [sys.executable, str(SCRIPTS / 'umap_visualization.py'),
     '--matrix', str(fm_path), '--outdir', str(RESULTS)],
    'umap_visualization.py'
)

In [ ]:
# ── CELL 16: UMAP panels ──────────────────────────────────────────────────────
for fname, label in [
    ('Panel_A_2D_umap.png',        'Panel A — 2D descriptors'),
    ('Panel_B_3D_delta_umap.png',  'Panel B — 3D Δ features'),
    ('Panel_C_combined_umap.png',  'Panel C — Combined'),
]:
    print(f'--- {label} ---')
    show_fig(FIGURES / fname)

# Panel summary table
summary_path = RESULTS / 'umap_panel_summary.csv'
if summary_path.exists():
    print('\nUMAP Panel Summary:')
    print(pd.read_csv(summary_path).to_string(index=False))
else:
    print('[umap_panel_summary.csv not found]')

---
## Section 6 — Tier-2 CREST+ALPB Validation

Two sub-sections:
1. **CREST reference results** — real ΔPSA from CREST+ALPB on 5 reference compounds vs literature
2. **Tier-1 vs DB cross-check** — does Tier-1 heuristic agree with DB 3DPSA?

In [ ]:
# ── CELL 17: Load CREST reference results ────────────────────────────────────
crest_path = RESULTS / 'tier2_reference_results.csv'

if not crest_path.exists():
    print('Tier-2 CREST results not found — skipping CREST sub-section.')
    print('Download tier2_reference_results.csv from the Tier-2 Colab and place in results/')
    crest_df = None
else:
    crest_df = pd.read_csv(crest_path)
    ok_crest = crest_df[crest_df['error'].isna()]
    print(f'CREST results: {len(crest_df)} compounds, {len(ok_crest)} successful')
    if crest_df['error'].notna().any():
        print('Failures:', crest_df[crest_df['error'].notna()][['id','error']].to_string(index=False))

In [ ]:
# ── CELL 18: CREST results table + literature comparison ─────────────────────
LIT_DELTA_PSA = {'CsA': 75.0}   # Witek et al. JCTC 2016

if crest_df is None:
    print('Skipped — no CREST results.')
else:
    ok = crest_df[crest_df['error'].isna()].copy()

    rows = []
    for _, r in ok.iterrows():
        lit = LIT_DELTA_PSA.get(r['id'])
        rows.append({
            'ID':          r['id'],
            'Name':        r.get('name', r['id']),
            'PAMPA':       f"{r['pampa']:.2f}",
            'Permeable':   '✓' if r.get('permeable') else '✗',
            'aq_PSA (Å²)': f"{r['aq_psa3d']:.1f}",
            'mem_PSA (Å²)':f"{r['mem_psa3d']:.1f}",
            'ΔPSA (Å²)':   f"{r['delta_psa3d']:.1f}",
            'ΔHB':         f"{r['delta_hb']:.0f}",
            'Lit ΔPSA':    f'~{lit:.0f}' if lit else '—',
            'Match?':      ('✓' if abs(r['delta_psa3d'] - lit) < 20 else '~') if lit else '—',
        })

    table = pd.DataFrame(rows)
    print('CREST+ALPB Results vs Literature')
    print('=' * 80)
    print(table.to_string(index=False))
    print('=' * 80)

    # Permeable vs impermeable ΔPSA
    ok['perm_bool'] = ok['permeable'].astype(bool)
    grp = ok.groupby('perm_bool')['delta_psa3d'].mean()
    if True in grp and False in grp:
        diff = grp[True] - grp[False]
        print(f'\nMean ΔPSA — permeable: {grp[True]:.1f} Å²   impermeable: {grp[False]:.1f} Å²')
        direction = 'higher' if diff > 0 else 'lower'
        consistent = 'CONSISTENT' if diff > 0 else 'INCONSISTENT'
        print(f'→ {consistent} with chameleonic hypothesis '
              f'(permeable compounds show {abs(diff):.1f} Å² {direction} ΔPSA)')

In [ ]:
# ── CELL 19: CREST ΔPSA bar chart ────────────────────────────────────────────
if crest_df is None:
    print('Skipped.')
else:
    ok = crest_df[crest_df['error'].isna()].sort_values('pampa')
    colors = ['steelblue' if p else 'salmon' for p in ok['permeable']]

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(ok['id'], ok['delta_psa3d'], color=colors, edgecolor='white', lw=0.5)

    # Literature reference for CycloA
    if 'CsA' in ok['id'].values:
        ax.axhline(75, color='black', ls='--', lw=1.2, label='CycloA lit. ~75 Å² (Witek 2016)')

    # Legend patches
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color='steelblue', label='Permeable'),
        Patch(color='salmon',    label='Impermeable'),
    ] + ([ax.get_lines()[0]] if ax.get_lines() else []), fontsize=9)

    ax.set_ylabel('ΔPSA = aq_PSA − mem_PSA  (Å²)')
    ax.set_title('Tier-2 CREST+ALPB: Chameleonic ΔPSA by Reference Compound')

    # Annotate PAMPA values
    for bar, (_, row) in zip(bars, ok.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"PAMPA={row['pampa']:.1f}", ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    plt.savefig(FIGURES / 'fig_tier2_crest.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── CELL 20: Tier-1 vs DB cross-check (tier2_validation.py) ──────────────────
run_script(
    [sys.executable, str(SCRIPTS / 'tier2_validation.py'),
     '--matrix', str(fm_path),
     '--refset', str(DATA_DIR / 'reference_set.csv'),
     '--outdir', str(RESULTS)],
    'tier2_validation.py'
)

# Load and display the validation table
val_path = RESULTS / 'tier2_validation_table.csv'
if val_path.exists():
    print(pd.read_csv(val_path).to_string(index=False))

print('--- Tier-1 vs DB cross-check figure ---')
show_fig(FIGURES / 'tier2_crosscheck.png')

---
## Section 7 — Summary

In [ ]:
# ── CELL 21: Auto-populated results summary ───────────────────────────────────
print('=' * 65)
print('RESULTS SUMMARY')
print('=' * 65)

# Dataset
print(f'\nDataset')
print(f'  Compounds          : {n_total:,}')
print(f'  Permeable (>={PAMPA_THRESHOLD}) : {n_perm:,} ({100*n_perm/n_total:.1f}%)')
print(f'  DB 3DPSA coverage  : {n_3dpsa:,} ({100*n_3dpsa/n_total:.1f}%)')

# Tier-1
if conf_df is not None:
    ok_t1 = conf_df[conf_df['error'].isna()]
    print(f'\nTier-1 Conformers')
    print(f'  Molecules processed: {len(conf_df):,} ({100*len(ok_t1)/len(conf_df):.1f}% success)')
    if 'delta_psa3d' in ok_t1.columns:
        print(f'  Mean ΔPSA          : {ok_t1["delta_psa3d"].mean():.1f} Å²  (std={ok_t1["delta_psa3d"].std():.1f})')

# Correlation
if (RESULTS / 'correlation_table.csv').exists():
    corr = pd.read_csv(RESULTS / 'correlation_table.csv')
    auc  = pd.read_csv(RESULTS / 'auc_roc_table.csv')
    print(f'\nCorrelation with PAMPA LogPexp')
    for _, row in corr.head(5).iterrows():
        print(f'  {row["Feature"]:<25} ρ={row["Spearman_rho"]:>+.3f}  AUC={auc[auc["Feature"]==row["Feature"]]["AUC_ROC"].values[0]:.3f}' if row['Feature'] in auc['Feature'].values else f'  {row["Feature"]:<25} ρ={row["Spearman_rho"]:>+.3f}')

# Tier-2 CREST
if crest_df is not None:
    ok_c = crest_df[crest_df['error'].isna()]
    print(f'\nTier-2 CREST+ALPB (5 reference compounds)')
    for _, r in ok_c.iterrows():
        lit = LIT_DELTA_PSA.get(r['id'], None)
        lit_str = f'  (lit ~{lit:.0f})' if lit else ''
        perm = 'permeable' if r.get('permeable') else 'impermeable'
        print(f'  {r["id"]:<10} ΔPSA={r["delta_psa3d"]:>6.1f} Å²  {perm}{lit_str}')

print(f'\nFigures: {FIGURES}/')
print(f'Results: {RESULTS}/')
print('=' * 65)